# Publish Pointset and/or LineSegments from KML/KMZ files

This notebook shows how you can sign in and publish Pointset and/or LineSegment geoscience objects from a kml- or kmz file to your chosen Evo workspace.

**Important:** This notebook requires Python 3.10, 3.11, or 3.12 (not 3.14+) due to evo.notebooks dependencies.

In the first cell we create a ServiceManagerWidget which will open a browser window and ask you to sign in.

Once signed in, a widget will be displayed to allow you to select your organisation and an Evo workspace.

__Required:__ In Cell 2, replace `"your-client-id"` with your Evo app client ID before running the cell.

In [ ]:
from evo.notebooks import ServiceManagerWidget

client_id = "client-id"
redirect_url = "url"
ims_base_uri = "url2"
discovery_base_uri = "url3"

manager = await ServiceManagerWidget.with_auth_code(
    client_id=client_id,
    base_uri=ims_base_uri,
    discovery_url=discovery_base_uri,
    redirect_url=redirect_url,
).login()

## Select Organization, Hub, and Workspace

This cell prepares your active Evo context for publishing.

It will:
- List available organizations and select one (by `org_index`)
- List hubs and select one (by `hub_index`)
- List workspaces and select one (by `workspace_index`)
- Create a new workspace only if none exist

Before running, update the index values to match the organization, hub, and workspace you want to use.

In [ ]:
# Select organization and workspace, or create one if needed
service_mgr = manager._service_manager

# Variables to store workspace info
new_workspace = None
selected_workspace_id = None

# List available organizations
orgs = list(service_mgr.list_organizations())
print("Available Organizations:")
for i, org in enumerate(orgs):
    print(f"  {i}: {org.display_name} (ID: {org.id})")

if not orgs:
    print("  No organizations found!")
else:
    # Select your organization by index (change the number if needed)
    org_index = 0  # Change this to the index of your organization
    service_mgr.set_current_organization(orgs[org_index].id)
    print(f"\n✅ Selected organization: {orgs[org_index].display_name}")

    # List available hubs
    hubs = list(service_mgr.list_hubs())
    print("\nAvailable Hubs:")
    for i, hub in enumerate(hubs):
        print(f"  {i}: {hub.display_name} (Code: {hub.code})")

    if hubs:
        # Select your hub by index (change the number if needed)
        hub_index = 0  # Change this: 0=Australia East, 1=Canada Central, 2=South Africa North and so on...
        service_mgr.set_current_hub(hubs[hub_index].code)
        print(f"\n✅ Selected hub: {hubs[hub_index].display_name}")

        # List available workspaces
        workspaces = list(service_mgr.list_workspaces())
        print("\nAvailable Workspaces:")

        if workspaces:
            for i, ws in enumerate(workspaces):
                print(f"  {i}: {ws.display_name} (ID: {ws.id})")

            # Select your workspace by index (change the number if needed)
            workspace_index = 0  # Change this to the index of your workspace
            service_mgr.set_current_workspace(workspaces[workspace_index].id)
            selected_workspace_id = workspaces[workspace_index].id
            print(f"\n✅ Selected workspace: {workspaces[workspace_index].display_name}")
        else:
            print("  No workspaces found!")
            print("\n💡 Creating a new workspace...")

            # Create a workspace
            from datetime import datetime

            from evo.workspaces import WorkspaceAPIClient

            connector = manager.get_connector()
            workspace_client = WorkspaceAPIClient(connector, orgs[org_index].id)

            # Create workspace with timestamp
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            workspace_name = f"Kml-file Workspace {timestamp}"

            try:
                new_workspace = await workspace_client.create_workspace(
                    name=workspace_name, description="Workspace for Kml-file to Pointset/LineSegment conversion"
                )

                selected_workspace_id = new_workspace.id
                print(f"✅ Workspace created: {new_workspace.display_name}")
                print(f"   Workspace ID: {new_workspace.id}")

                from evo.service_manager.manager import _State

                current_workspaces = list(service_mgr.list_workspaces())
                current_workspaces.append(new_workspace)

                service_mgr._ServiceManager__state = _State(
                    organizations=list(service_mgr.list_organizations()),
                    workspaces=current_workspaces,
                    selected_org_id=orgs[org_index].id,
                    selected_hub_code=hubs[hub_index].code,
                    selected_workspace_id=new_workspace.id,
                )

                print("✅ Workspace selected and ready for publishing!")

            except Exception as e:
                print(f"❌ Failed to create workspace: {e}")
                import traceback

                traceback.print_exc()
    else:
        print("  No hubs found!")

## Convert and Publish Kml/Kmz file to Pointset or LineSegments

In the cell below we:
1. Specify the path to your kml/kmz file
3. Add optional tags and upload_path
4. Call `convert_kml_kmz` to convert and publish the corresponding Pointset and/or LineSegment objects to your evo workspace


**Note:** If `upload_path` is not specified, the generated Evo objects will be published at the root of your workspace. If you don't have workspaces available, use `evo_workspace_metadata` parameter instead of `service_manager_widget`.

In [ ]:
# Install local package into this notebook kernel (run once per environment)
%pip install -e ../..

In [ ]:
from pathlib import Path

from evo.data_converters.kml.importer.kml_kmz_to_evo import convert_kml_kmz

# Path to your KML/KMZ file (can be any of the individual files or just the extensionless name)
# Using relative path from notebook location
filename = "../../tests/data/KML_Test.kml"

# Tags to add to the geoscience object
tags = {"Source": "Jupyter Notebook", "Type": "KML Point and LineString objects"}

# Upload path in Evo workspace
upload_path = "kml-file/imports"

print("Converting and publishing KML/KMZ to Evo workspace...")
print(f"Kml/Kmz-file: {Path(filename).name}")
print(f"Workspace: {selected_workspace_id}")

# Convert and publish
results = convert_kml_kmz(
    filepath=filename,
    tags=tags,
    evo_workspace_metadata=None,
    service_manager_widget=manager,
    upload_path=upload_path,
    publish_objects=True,
    overwrite_existing_objects=True,
)

# Print results
print("\n✅ Successfully published!")
print("\nPublished objects:")
for obj_metadata in results:
    print(f"  - {obj_metadata.name}")
    print(f"    ID: {obj_metadata.id}")

## Convert Without Publishing (for testing)

You can also convert the kml/kmz file without publishing to inspect the resulting object.

**Note:** When `publish_objects=False` and `upload_path` is not specified, the generated Evo objects are saved as **Parquet files** in the current working directory (the same directory where your notebook is running).

In [ ]:
import json

from evo.data_converters.kml.importer.kml_kmz_to_evo import convert_kml_kmz

# Convert but don't publish
kml_objects = convert_kml_kmz(
    filepath="../../tests/data/KML_Test.kml",
    publish_objects=False,  # Don't publish
)

# Inspect the KML object
kml = kml_objects[0]
print(f"KML name: {kml.name}")

kml_dict = kml.as_dict()  # Use as_dict() instead of to_dict()
print("\nJSON representation:")
print(json.dumps(kml_dict, indent=2))